# Z-Image-Turbo on Kaggle T4×2 GPU — Production REST Demo

**English**

This notebook is the reproducible Kaggle entry point for running `Tongyi-MAI/Z-Image-Turbo` as an **API-only** image-generation service on exactly **two NVIDIA Tesla T4 16 GB GPUs**. One resident BF16 `ZImagePipeline` spans both GPUs: the transformer is placed on `cuda:0`, while the text encoder and VAE are placed on `cuda:1`. This is one logical pipeline, not two independent replicas and not a 2× speed claim.

The workflow validates the attached Kaggle model, reconstructs the audited server payload, starts an internal worker plus an authenticated FastAPI coordinator, and executes deterministic acceptance checks for the qualified `safe_512` and `high_768` profiles.

This is an independent engineering qualification project, not an official Tongyi-MAI release and not an SLA-backed hosted service.

**Tiếng Việt**

Notebook này là entry point có thể tái lập trên Kaggle để chạy `Tongyi-MAI/Z-Image-Turbo` dưới dạng dịch vụ sinh ảnh **API-only** trên đúng **hai GPU NVIDIA Tesla T4 16 GB**. Một `ZImagePipeline` BF16 resident trải trên cả hai GPU: transformer nằm trên `cuda:0`, còn text encoder và VAE nằm trên `cuda:1`. Đây là một logical pipeline, không phải hai replica độc lập và không phải tuyên bố tăng tốc 2×.

Workflow xác minh Kaggle model đã attach, khôi phục server payload đã audit, khởi động internal worker cùng FastAPI coordinator có xác thực, rồi chạy các acceptance check xác định cho hai profile đã qualification là `safe_512` và `high_768`.

Đây là dự án kiểm định kỹ thuật độc lập, không phải release chính thức của Tongyi-MAI và không phải dịch vụ hosted có SLA.


## Before you run / Trước khi chạy

**English**

Before running this notebook in Kaggle:

1. Select **Accelerator → GPU T4 x2**.
2. Attach Kaggle Model `dangkhoa2016/tongyi-mai-z-image-turbo`.
3. Select **PyTorch / default / Version 1**. The expected mount is `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1`.
4. Keep **Internet = ON** only if you want to exercise the optional Cloudflare Quick Tunnel step. Local qualification uses loopback HTTP and does not require the tunnel.
5. Use a fresh Kaggle session and run the notebook from top to bottom.
6. Do **not** copy the full model into `/tmp` or `/kaggle/working`; the model stays read-only under `/kaggle/input`.

Qualified public profiles:

| Profile | Resolution | Steps | Guidance | Role |
| --- | ---: | ---: | ---: | --- |
| `safe_512` | 512×512 | 9 | 0.0 | default deterministic acceptance path |
| `high_768` | 768×768 | 9 | 0.0 | qualified higher-resolution single-flight path |

Unsupported by design: FP16 fallback, CPU fallback, 1024×1024 generation, multiple concurrent GPU pipelines, and full model staging to `/tmp`.

**Tiếng Việt**

Trước khi chạy notebook trên Kaggle:

1. Chọn **Accelerator → GPU T4 x2**.
2. Attach Kaggle Model `dangkhoa2016/tongyi-mai-z-image-turbo`.
3. Chọn **PyTorch / default / Version 1**. Mount mong đợi là `/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1`.
4. Chỉ cần giữ **Internet = ON** nếu muốn chạy bước Cloudflare Quick Tunnel tùy chọn. Qualification local dùng loopback HTTP và không phụ thuộc tunnel.
5. Dùng fresh Kaggle session và chạy notebook từ trên xuống dưới.
6. **Không** copy toàn bộ model sang `/tmp` hoặc `/kaggle/working`; model được giữ read-only dưới `/kaggle/input`.

Hai public profile đã qualification là `safe_512` (512×512) và `high_768` (768×768), đều chạy 9 bước với guidance 0.0. Chủ động không hỗ trợ FP16 fallback, CPU fallback, 1024×1024, nhiều GPU pipeline chạy đồng thời hoặc staging toàn bộ model sang `/tmp`.


## Workflow map / Sơ đồ quy trình

```text
REST client
   |
   v
FastAPI coordinator :8090 (CPU, Bearer auth)
   |
   | bounded single-flight queue
   v
Internal worker :8101 (loopback only)
   |
   +-- Transformer ---------------- cuda:0
   +-- Text encoder ---------------- cuda:1
   +-- VAE ------------------------- cuda:1
   |
   v
One resident BF16 ZImagePipeline across T4 x2
```

**English**

The notebook intentionally separates **runtime qualification** from the optional **public transport** check. A temporary tunnel failure does not invalidate a healthy local model/runtime.

**Tiếng Việt**

Notebook chủ động tách **runtime qualification** khỏi bước kiểm tra **public transport** tùy chọn. Lỗi tunnel tạm thời không làm mất giá trị của một local model/runtime đang hoạt động đúng.


## Step 1 — Runtime and model preflight / Kiểm tra runtime và model

**English**

Verify that the live Kaggle session exposes exactly two Tesla T4 GPUs and that the attached model resolves to the expected `ZImagePipeline` before any service is started.

**Tiếng Việt**

Xác minh session Kaggle hiện tại có đúng hai Tesla T4 và model đã attach đúng loại `ZImagePipeline` trước khi khởi động service.

### Expected evidence / Kết quả cần thấy

**English**

`PREFLIGHT=PASS` plus the printed Python, PyTorch, CUDA, Diffusers, Transformers, Accelerate, GPU and model provenance JSON.

**Tiếng Việt**

`PREFLIGHT=PASS`, kèm JSON provenance/runtime in Python, PyTorch, CUDA, Diffusers, Transformers, Accelerate, GPU và model.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys
import torch, diffusers, transformers, accelerate

MODEL_PATH = Path('/kaggle/input/models/dangkhoa2016/tongyi-mai-z-image-turbo/pytorch/default/1')
WORKSPACE = Path('/kaggle/working/Z-Image-Turbo-Kaggle-T4x2-REST-Server')
EXPECTED_MODEL_INDEX = MODEL_PATH / 'model_index.json'
EXPECTED_GPUS = ['Tesla T4', 'Tesla T4']

assert torch.cuda.is_available(), 'CUDA is required'
assert torch.cuda.device_count() == 2, f'exactly 2 GPUs required, got {torch.cuda.device_count()}'
gpus = [torch.cuda.get_device_name(i) for i in range(2)]
assert all('T4' in name for name in gpus), f'T4x2 required, got {gpus}'
assert MODEL_PATH.is_dir() and EXPECTED_MODEL_INDEX.is_file(), 'attached Kaggle model missing/wrong variation'
model_index = json.loads(EXPECTED_MODEL_INDEX.read_text())
assert model_index.get('_class_name') == 'ZImagePipeline', model_index.get('_class_name')

runtime = {
    'python': sys.version.split()[0], 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'diffusers': diffusers.__version__,
    'transformers': transformers.__version__, 'accelerate': accelerate.__version__,
    'gpus': gpus, 'model_path': str(MODEL_PATH),
    'upstream_revision': 'f332072aa78be7aecdf3ee76d5c247082da564a6',
}
print(json.dumps(runtime, indent=2))
print('PREFLIGHT=PASS')


## Step 2 — Materialize the audited API-only source / Khôi phục source API-only đã audit

**English**

Reconstruct the exact embedded server/test payload in `/kaggle/working` and verify both the archive digest and every allow-listed file before writing it.

**Tiếng Việt**

Khôi phục chính xác server/test payload được nhúng vào `/kaggle/working`, đồng thời xác minh digest của archive và từng file trong allow-list trước khi ghi.

### Expected evidence / Kết quả cần thấy

**English**

`BOOTSTRAP_FILES=<count>` followed by `BOOTSTRAP_INTEGRITY=PASS`.

**Tiếng Việt**

`BOOTSTRAP_FILES=<count>`, sau đó là `BOOTSTRAP_INTEGRITY=PASS`.


In [ ]:
import base64, hashlib, io, json, shutil, tarfile
from pathlib import Path

WORKSPACE = Path('/kaggle/working/Z-Image-Turbo-Kaggle-T4x2-REST-Server')
PAYLOAD_SHA256 = '8f01ff33b84bce59e4760a6865629c6016392e0a90b81016989bfcb53cc15652'
MANIFEST_SHA256 = 'ae4b13b7220dd41a20b55aeb56acd6cb9fb57bd4e2baa8fcbb6a4605969d63bd'
MANIFEST = json.loads('[{"bytes":13,"path":"src/zimage_server/__init__.py","sha256":"75ad4f4c2943325f88064ad2beae890b59674be2bf0404b30afc5584dc4de4bb"},{"bytes":4913,"path":"src/zimage_server/app.py","sha256":"cd91cc210e24c8fc1cab1cf3ef11dfdd8bb4eb9dc4dd82f5413aac92bb882c8b"},{"bytes":347,"path":"src/zimage_server/config.py","sha256":"eed00972bfc315774cd2d71db20935ce1c6f0048d58fdcd2dc9137ba58b94a72"},{"bytes":928,"path":"src/zimage_server/internal_client.py","sha256":"ab65fd699dc0c04d9f58a3686cc8732206a6a8a7cc3210fc85bd97e8a17d3606"},{"bytes":3128,"path":"src/zimage_server/jobs.py","sha256":"ef3b335d6d129e9fbcaa77cd3716e329d009fdb67c0074121bf7ee92ec21c85b"},{"bytes":4296,"path":"src/zimage_server/runtime.py","sha256":"675eafe52416c680e0a2f46474ba51c8eedc251aca5cc443c292c41110fe86d3"},{"bytes":776,"path":"src/zimage_server/validation.py","sha256":"7355f1a3121f1e664f5fb0cf1422a0e20c8ad02f978d5959a5186a1e81692922"},{"bytes":1639,"path":"src/zimage_server/worker_api.py","sha256":"ebc94057c0191d495baad119081b6f08a4a71d08d1af24f5978c93dea15ba9d2"},{"bytes":3118,"path":"scripts/endurance_queue_acceptance.py","sha256":"1d5b550e212371523749339ec0cac76202e480ed2198713cf338ccc576ac9c65"},{"bytes":1847,"path":"scripts/error_boundary_acceptance.py","sha256":"8ef6edeb6041f54d9496f804462b957a3c94cd2439ba06b0815891fe3bdf70f0"},{"bytes":2449,"path":"scripts/finalize_evidence.py","sha256":"5b9787a04f05e904af985394e5183cc14f132ac36a63946111ccf71703d5cda0"},{"bytes":2351,"path":"scripts/golden_rest_acceptance.py","sha256":"63d8719c4cd3c849653fad768304191381bda6b15c817fab8f3f97b5662585ae"},{"bytes":1153,"path":"scripts/local_control_acceptance.py","sha256":"de226083874f29f42f1cca90ff7c7a9ed60f143ff6992b43a1622214a77350aa"},{"bytes":3196,"path":"scripts/mixed_profile_acceptance.py","sha256":"c07760271b4d711a45e03c4d13a58666bbc29acf8e496fc5c476fc20cacb2a02"},{"bytes":2624,"path":"scripts/public_tunnel_acceptance.py","sha256":"ce8b0ede3fc8e31f0b38fa02808fe9eb5ba17adfeafa9eb4ac29bf582122cc22"},{"bytes":1999,"path":"scripts/qualify_mixed_existing.py","sha256":"fe6e1802288b7e92161593a8ae11faf55009512d189e07285bccd7eacc5fb14c"},{"bytes":1654,"path":"scripts/start.sh","sha256":"fd0cf4974dd5171e3e17363a16e42d2b3ca43765434df5c78dc0c64bba02dacf"},{"bytes":403,"path":"scripts/stop.sh","sha256":"6075f5c3dd4c7591fb0e234a0d3b0ec178bd294d3b957bdd1fd91cea8fe2daef"},{"bytes":264,"path":"scripts/status.sh","sha256":"adc8bdfddab5f53550151aeb5fd57980d4ced498c9382bcdd8e57d2c45d9a788"},{"bytes":1022,"path":"scripts/start_tunnel.sh","sha256":"6e85fd13630818a2161d7a3265128e2cf264bfbc6d9a28bf5b820238ccb8d966"},{"bytes":1118,"path":"tests/test_api_jobs.py","sha256":"d4c276f9d8fa223fd37722d942e7f595ec96c03cfda9cc58ecedb5a29836d7fe"},{"bytes":1214,"path":"tests/test_contract.py","sha256":"a415472193557627d315154f76aa8a0cb0bec74063f8de776f574b39202dadf9"},{"bytes":1132,"path":"tests/test_jobs.py","sha256":"c2b664ff7e0e1973901f3b2b436fdd293ac205a799a8d7d2735eae662ef60777"},{"bytes":488,"path":"tests/test_profiles.py","sha256":"70d0ca794d43ba888aa760810fcf8c401a87398e12c080bef85c12d854a6fb53"},{"bytes":759,"path":"tests/test_root_metadata.py","sha256":"8bbeb036fbe0eab31e083ff6e120788fc8ed16e74d34c21fb42dcc2b2f25603f"},{"bytes":920,"path":"tests/test_validation.py","sha256":"7d290a312d25d13e3ae2cbca06f2373ab45adbada110ef51a9e91086059274fc"},{"bytes":39,"path":"pytest.ini","sha256":"0922d82a60b616ffaf801784738325d7f1925ec773c99135f64767d14166d725"},{"bytes":689,"path":"pyproject.toml","sha256":"33105bcd4544e466fbb671dc32edadca7b5e61c23a5e7fd795e3f517461bf4dc"},{"bytes":3871,"path":"scripts/api_contract_acceptance.py","sha256":"2c33e8ea927c67eaa5e3a3a51c1c59c3282f13782356e8cb3c420506e9fc8a61"},{"bytes":5259,"path":"scripts/verify_evidence.py","sha256":"bf1394291d78d4bd98c34b6ad34c9dca0c390d9a42125db6ef66692242a7d201"}]')
PAYLOAD_B64 = "H4sIAAAAAAAC/+1923bbRrLoPOMrEJyZEemQFEHdbE7oNYpNO5rIkiLLmUlobiyIBEVEIMAAoCxFo6fzA+ej9oedquoLugHwIltWMh5yeVkAurr6Xl1VXVWdxIPN3/yJe+E5iRdfefGm4/ihnzpOY3rzp4f5NeG3u71Nf+GX/9ts2jvZM363t3aaW38ym396hN8sSd0Yiv/Tf+fPcdwgcByzY/b6xp/Wv/+2X1JY/+50+nBLf4X1b9vbu3Zu/bd2ttbr/1F+/mQaxak5nrgDgz9HiTGKo4k5ddNx4J+b/PMJvLKEQRSm3nWqpLnJTTjgnyduCLMpNhjsyIX+nfoC8KU39cJhUjNfwff9k4Oa+Z3nDr0Y/p6dnXSvB9409aNQy9uIvWQahYmXCCyv/MA75R95XW+Gbpj6AwHxrZt4b6KhF0BJvhcMa+YI/zhXbuAP3TQS1WtApUf+haxe99X+u8Mz5+T0+NXBYbdm8oe3HPqX6FxW4h/R+RvW1Jr5w8ybea9mQdCNY8RtDAI3SczXXujFLjbo1Pt15iVpRdar2jZM+E0B7TRtm0kaAwmmqlYmfugEXniRjjt2zZy41+Jtu/lst0rZEs8btk0/TCHTdktgGkG3CFS5lhgE8/dcJ1QsVr7FsP6daj3x0nE0pA9Db2SGUeicB2546TDYyiCA4QMUM1ZW1aw/x7+sPfjzMVPKQBqQ4k8r1SwVf7HrJ575IwJQh4l6mBNYjZT53DOpUF4zyuSlszhkaBc0BzthUXsuw+hD6HDAlRpDyVQrP5QTYlmDZiEVZOo1KrbDwDo57iwdV/C/KPZ/ownDxvHf5lEUejCcbJVUANidBWkHv1apsvjE6uJdT71B6g0BOkoaXnjlx1HYuPDSivXzwZv9110H1ptzdvx998iqmRavER8srWwziosfG0gn0+SDDzW1vvXc2ItNSxlX1gXaKq5AlnSWOAOY8DB5YTIPvdT1g4418ZPEDy9gmAlPGl16Ia9QMptOA5+aoZXf22v31RrL5vLKIgWD1TyZAkZn6F/gahO4ahL6XhXeyirshzTL8hVmwzeIPTf1HNg4K09qJvWT8yGKL724bZ5HUQBNOYtnXs2MZyEQBBo9eIEh9UMvSfg7LvRfkY44if+bJ1c3jTKnlm3RAWoZphsOOWbTT5QJgT9GtgCVF4du4AygNwAtp2CQybmQJKpmMnwOVuwm6yaGuZODVqYzbwaCyGcYFA0bgfOdAQAz2lnhfcL+5Huho79WC233AhhDbDKnCGUbESbQd1r/gT+CncMNKzBcbdGxyrS4QZqiLn9Ra+hbnGd6/yrNagyCKPEqVVYTwI4EnaGvpH4aeB3r5/oBclr1s1l8HpnfuxcXgWeebV+3zNPu2zPzLTFgsDjh/wT6uGM1G3ajCR9EpTvioSraC8waLfFNgPLDQTAbeiC/Oclg7E3czisX+qcqaV8cRakD9NAFcumqRJnTpFutXRYyhP7As9qm9VudeMR6ijWvX1LN6ynU3KrpedKbKWWAPTutw+adT+dNQxDeOD19GA0STNykh1zi2HODdEzJ/DEHQJON0tlTLtkPRxGlXtmb9Jyl3xU6lJeQ9R77UNZtFqMbiDq6tIqoeGUAC3I/XjjwvaTT47xQhah/ta8ME4Kr5eizsDgDl5GyHZWU8ZWDc5mWkjdUNid/pCzi8rJEm0VfM9LG0WIX+OEvRGuhI0QmrIynEohKNc8uEAjrL4a5+vFtJFyFHZe+FsZGToVVRweh1cHhOzzSv9vQnQDlvnLjpMI/V80RblDwvSYgVTai4afeBHoj6yo+PFpfFYZFECJG/7KxKKFVdyWzdaV1bQpuCue1eFQHmj2os30aJaJLEWuyme0XybIOpo1Tjmar2cp6nGPxKjHjottFxvoRFguflXOXihwT3I9FBj6ADzSvF1UnjW90xCCrOD6yUWJ/SmbnEz8VndiYoBjiDGcTYM8zNB5VIifQwPaJCfer+HbrmbIg4wogAJaVGBJ4LM5KfwhzilUaZllGUWn7HyrzET6KNvAvCA8SkZKAr3dlax0luM1bVszd6oseMlRYJiYn/A7TDQiFMpjYIFYhbUYS1EfVY7u5ndUDMFMlRtEsHBZlFyxlaecyGnAPuorg606WJfTEGuibX3VMC0WbwEs9Swg8CCPoSgJC4X0Jy3bzWbEuspSsOqgGgj5B9U+F1YuX12elR7N0OksdBGNyZWFrx6SGnzgkcVc/ocdYUSaXHovzUtULVbBUkCe8oe86yJWCEIczbHMKOVEdFHi4K3eocvjEWWqOCuY1SHeMi1fku6rxxet/mUbsIVXAy85/UNmbO/9p2btr/e9j/Gg/RnGQdGWZilV+glXwd/lSAfDfvLCDPH+VKzlPuOKRVs8Hf5iOSXdBr2PPvxin2XuSetMke72Y+UM3HMDUG7iouhwFkZtCgTnlJSxBK3FHnrNjtyxDsM7IbxtMThVpbVGZCtWjA99qvA7smcrvAN3TS+40G80qkwWtMUA7e7tPC8jgm0RGz4uR3Rn/oes/pyx6AEKwZP3bu9vN3PqHwVqf/zze+nec0Qx2Pc9x5IFOCBs3E9wMcS70SxKF4nkWB4F/3vDouEP/xllww/jn8en33VPn2/23tILHaTptb27arb0GrJCG3X5qN22Lq1BVZWGFVJ5Df5AymqJJNqh/zpXUgNcImMqKWuDXmQ7ITP2JB6xDZ6eKgow4TSpVa2AbG0CFhhUBVjUUsUjyKAWJqKAWIdUb8EPURfA+sm6RCyExqOE4yHE4zl3bvIUPd6gzon7QlazIfLapJ3JdwlsO3ZrrCnHOJOuV7xMhSyuqL6TuHWo4CoMJFlptALsO7FfFmqWj+lOrmkGP6SQi6dxaL1DNGqb1M67yc1HZPqCKbyI66y7LxQ5hOtbJ8dszXnT1foPL37PR3Go2P3k81WncICa0VNw9j4Y30NfwrYHDC/L80FO7p2YShqQDoz8N3IF23EMM7uksxHqzw6GREEGwRJoADcSGkwFLurMUMfnx6T8KcQ9rALCE/u829/L0v7VnN9f0/w9G/0kRI17SMVNvXcgPML3lTgAMkTAVmN5IltLzpvjOUoAY4gkgT3vhBoF7Hnjy6FxXQFXU9SOOzhlzysCzA6W2VCgIQ7ZK4gUjcfjWlkX1ekhP+zWT/5l3/paRAcTTcNQDsZAfL2WphKHN+qpBjeilMxCpe0ka86L4qW4fUChgikacYXKHJOfCXtPJervxLSoLvOFbb+JOx1HsVfRqA42384hwQbNdRK0EcM93OcAgGlxqhR3Ch0K9WLIGd0ZPFVhIF17aEciiaY0U3x2LEZg6VKTOtcdQC9ebREKSKCuBHTiL8zQcUK7AZMM5dW+Qrivbo3ZuL7WfIPVfOBb0C07KBv63DfR77F1L0DD6gK2B6dXA/yqqWmEQxYhkFe0kq13qDR03hRTAiipLVkt4F7O/wj9Va6ZQpLRNdgIseQVdb0+7YjZC+j6XDXGP1a5PRwdY78IJiz6tGu7g15kPM+gcscIw8tNCHf/CwvUKNKbRtCK6iCwUSjQ9uXVtCU6G+hGVaiNIs6plq6qBaiboVhfmgCyn0K3VglaIgRrKcYKYQ6qyUTBYfHW2V+t+rjBUOqFMZ6hURtYXc1alNjF3jk0sseuX1FPyP1xazjGGLLM7DHwyG6FJDcssSqPQHwBj/bXInjVvjEdSBcBvJJr5DZ7X1IUq0mwqfO/dsEmwAIeiBPWRu5b6Sbla7krQK1pq9Ts1MglgBCoggdh5Hu2M9YxeJzkc7HQf252zuyk9fJkzbfVVwZlQBoVrQsdBG1wpZfwl8sOK4INbSiWJ6s6rJBtpJLgL5jCrDI7sPcdUzZ66yaUzBLAckmxwSk6qJEmHuhSGLkvCuS4XexlMT9AUwGkhNeTrX5sJ+fFaidSV01tlisJug0wB6qbvkznmm0Z/zjaUdR1uF3KoGP9R3hcP1h654FbOPQKuKxmv1qJ5OMQJA+1l+KjlXCqMP2gPMDLzqM1nRRKPJgxqiroDoIATL0nwjI12Bkq908qAugCzezNvsWasQOwFnpuUVXHesl5bzn+p5z8xE7UeUAWwxP6/uWXn5f+t7db6/OePJv+P3QT9AcrUwST8z3UakFBRPOAuBEN/NJrBhJPnTT+TZeSJP/WQ5yy3zs8Z43O7b9wAOAD/4jk0oQ3jzfHL7qFzsn/2nTjAtjaZjdOmHwJbtkk2MMnm0A0vLseR22rau5vAAV/c+PWJ69c166jN6Q01YJMbYW/aIKYcvzs7eXfmnB4fnxWKQFEXWIFNzeqzzqw+62j1WUerzzqz+txkZ9sJ4HzTPdt/uX+2/xBYhbEnoO3+66T74qz70nnZ/fHgRdd5s3/CNpjYDZNRFE/IqKsJ2woazzpMAYyfbPh05eIuY99JhQsbL66WmaNzyatOpjC6UCSJOVoC8i9AfmC0h0kpwNBDgzVn4k5FsixxGnujAA/lMrZXF4e49Etj1xjMhi6aIrhXrk9aIBB1olhN5EUNImgayUGtMoMFTZ9rvT5555xt/6vlnHZ/eHdw2n2pCLC4WWOjekoZwF47vBxMrvjMStBHGQdG48KrtKp9tQFueFOxzrYt4YWAuaRhofiQVJfWdFSsaptZLSZ3VsFuI1s+2GVDP2adhUkVZWltAiNCtmR+OPSuG6T6ry6299C7j+E6OMKV9Obg7duDo9dF245b62I6o+5CvQv9rYmCyfaEMUFZvVDBMhTmyOckLNu7wjiXLMHxWGDOpJGzSnDb2TSrqlatMXO4YBumF4/YvAGuvFo69XUq10A65gBmWILwOqzoImSuNVoiNazDppRoWw5CLpmOde4GeD49zNlDo+pw4k2i+KZz24ROsrdf+9+CZG0rz9ZgOsMObD3F9zsdATDVbkDDnDhRGNyQNi8DqS5c5SV9ZtZFny5Y/iQAyk5tjEdOlqxN4XzWrzpmCQ1cYc28OyrLZ97mCrgrmbRPnsiJI2er2g84a/N9g/NWIhUASjFG0TBWqolKTwvV/qCJuMB2rWxlHh2fOYfH+y81uiaMmTtyW0b5pSeNNPt9pTOEPM6kcV1Cz3xV8iRQr59CP0EmBAo69dxLPn8dFNgSoKPZkZ0H6DwpKzMwZVXyrotQkc8wvxZfKqyvOxaW1W4CLZu44QwmOhqVVvyQlF49ZnHar96XHBBHIakKjIa+6pkjXEd0JTrn9fVFx01QeD832Gstp7RFgxUBQW86QDibALUeebFHZitkyCLA6U0Hzxm4CEj9cy6L6M2OfFIIQ4PZh/eayi6XVee+JALHHqB17q9C/0uje94Lpt5t2bAojFxjcol7HTqShWnCaJrpXfswX6PL3IGFxqvdJ6NiNQlVV9nITTQVyBbIXWOqqpOoVY3EvfIqCooaLp6Jm3asE23vZH5wUADn3RvJ2G3t7KpZ6UTbOb9JvaRSpYMR7jyXYXFHKR14zVlKUi11m3fGkYu8rdCAnE9OZtStk49a3heJ7Lvzqy8HRYOcuSuUTX2LjbwCVLaCLFoFClDJqrD0+a8AL1oYJTxJ0UtJXwnU7ty3fHVpXPGYiQYvl6qa6DI2SfmS78Opfw2bFC0qAoa/OQhOcxmJBRj2UA5Ecwdg6G++Jxbscqpnlniq6JJRYaFwzvND7AMRQCGmoli1sFmKTnJDWJ2dFrCHJOLgwZewcslv31z3yCzdsSf8Qc6bWK6G4o5b6lM3Qi721r+DBt8WdG2WGyBHlcqZrmx5ojsFBOx1uYHja460OPMRCIA5+WlPXVQNyTIurQqhWlCfDNP8OuXUmXkGQZkja03eQ+n/MrXKQ6kAl9l/bhf0f9vNvbX+7zHjfwBDOL3Bc5Rwyi0Ry7kp4UfvZIbeykfF3DtHEP2R4FzQSAUksYqOqYCk4KivBFcYWbNQuv8PgUcM8RAjAZKalSGEMTdGziWcNtwEHt0b1hBULMJcTyvW6etvhccK13oArJ/gKQ7IVZCl2gBCV1lUHSuMwjrLYNLmmYhQBikKA7TLIybgIYbC1W4W+rBvAVsbRDHyrkgcoWD2mYCBKI5dkA7qds3cgs3KBT6y06w26CswztUsFsDQ/KZj2o0mKmd0xPgdVtSiyg89IUWyEeJ1L+xekrfKDZxRZKxyQ6mATPyQ7wTYRHirqBTfgj1BTXavc8mei9mzDsUPOgh0BjEtCpdpaV0Cqdo7g/vv3T+K9J+bYmPgnceh/1v29l6e/tv23pr+/8HOf6JkrvGnavD50bGhspBQxVBQc+M7iZMifmqpHydxZZZhOCK5o6eAFOuI2ARqFAdSMfjkKKgZ5cErN+djJnpzTSbFMck/aTGtEP1JD+tUiOhk6Hq0XHQgdiJicAsgEOdj0bwSDwrWYN3GATNB7R1SUFoFUGozAgrEzKS+uhAnc4RYzXuiLH/O7kKAfIpthGGUBohZLThMrmeNLFiM8BNdGO3lgLtV8SkxJ6iLURaApBh8pBh4JN+Ddzou4ehcCC8CDESh87+So7d6tCTVw9pR4m/MCxfCl5E+w2SleSAJ6azCKq7GgWjrS+uRWsNZRFr2y+1n7+WBzV1CzmeJcEjXVm1mAAeN12I3GMvs38q/A69YmWsbXdCEyIUvx0AzOysYGrEeysyK1oL2H5X/G8T+NE02vXA4i0lnyZwJ3AHOWPzw6VzgUv+fnd28/+fWOv7no8r/BWOecp/O+SY+8708m8+alkFhBz/RlkXweJvAMzo8Ah87vSBdb1XEmjT235HBz621rwYQRL5OxC0Ekkc1ujP+8fb4yPmuu/+ye0oe5U+eYG7YoJZ7N6In+psTtNCx9s2BH3oTVBCbxEr7CT5CL8VufMMkVvR2cs+jK890zYEbTMxoAOKr6aZmMgshA+yKH9x4Yl5FwWziQVMwP50bA0MYY9GQMx6M/RSEa2DWKYJgwArCM2b4iux6zUQP9uCGbyzeULjYSu8ZGQUx21NuxVlj22SNyiLq4B8t9k7mbn8nuNL5fqjS73RuTKic86nYVoQDarUmPU7VoarpHqVVYwX30cx11J7nOYohEDAwIk9osN3a7HTMVrNlLPct7Vl0om1wdiV1srhBknNbsbcolg5OVGEdL7oB5+dDtXeBl6zh0xFsT502pIHHp0wJbzcxrif8v1vtG4qDhV01UCPmuxj9sie7wudI4AFxQBl9g3d6MptUflFFAMWanuWhHAxplVRbNfEqcKCq7pe8z0jmoiXQ1RS79rsibsNgBz8Ji4qu11fYui3ysdluNY15HhcYaadjqh2i2miws837ubycAx24VJEUavR8jjNPmd+LWiF9PJUFolezo/oJkBmKkZ0LJw2gnMBdkFu5QZ05jb0rP5oBqRrM4pgCkobmb0C6eY6ayNqz232+bnjBPEPObeJ5R+LMGeDXzEqhNKgFzLUJEuZsfGFUfHHgx8c4zqxVmDcCj09LWUWrFAWpcBFkREBVSvLYZOLUWk3SnQZZv6rfNOCszW05BHEJoNoFHFLvFWPhYTNUtPhZq4g4bcYmsWc1+WI6a2bHg9k5MUDrR8f9HsHi3+zMcQEmcZicIWJfVsFj36NG9sqYFtfIXtKywmnrgoblYAt1WhGXPR/XHSwMhY5e98q7n9Y7XwW9Zn8uFC6ra1xUHLZaRG6vhNxeCTnjLGGpY6Q5L5Nn6jvklrnQNuDW4qSfyfq4BnrqNpLfIYCsYMQ0so9g5d8tMSyYxmS58rmKFAVYP7zrvus6b7unB/uHBz/vnx0cH3VO9t++lVWwXh382HX+cfyt0z16+e50/+hFNwfwpvvm+PQn5+3Z/rcHhwdnPyGrJ0DW8vIXK//j7u+cY3QBF0nGA4r/S89/tnfz5z87rVZrLf//TvL/CrGdvjQ9wGryPhPrSDfNxVQpld1XuvtUWZj/LRWD7xNSaQUZUbeHU+Xi2gOFV+LYRRwkBW1SKcZcgs3OGIy9wWXCYpiw5564U4LiyjK9Nw6UotmwYJtNrUyzYc9Ta9TMVYJbVWXJ/HKIByp53mTlpdQTj3x262zmrzRzs6qqN7mUVxUZypVqyvQQAvEsxMs2IpJG7GZr+2P6gUIvUmaJXvCM5YMMHCLyi9tNOw+nD4mE28rD6f0h4FqtPFyxeQqssdqyF+Hosghqy/WTVePTFjCViXJsyUKVyi4A6fEjsj76hqDCwigw1Lh064JBWMZPs36DJrGHGj+Cy+QlfPsIplk5PkPst5dt8wpHAtnly5p5hRwzK1JG1i8vWza4lI3unp4enzrfHr87erkPvPD+ixfdkzOVV/7i+D86wfN/8xwx4I/m/93c3m3ZBf/vne01//fY/N98rk71RHbI4c9xkOdKouAKduQGd3np2X0DuT9uni+JhziCIALE7sjI4rvnddBJpdKFzBSSW2HrCuQBrSWklA+FEnKrXO43Jv41eWYxIHqrwz5W39t9in8F2HR2DlunhGOv9RQDrQR1RSbi4CzqY1Z2GYk0LqIAOkECsVcqHHfBbVk2GqXE7iCVkMDe1sXHOiN8dSJ8Ioeg4AxlD+/WQYtbrpElSiVAqMWoWiTVAeqAFgCzZqsgpIgmGKCmb78/ODnpvgSG2B+VwEp0muJWwLG/DuxgHPQbbzr2JsAFB/VfZ/7gUvQ2QDy3DIyLVYKHR54nkwo3sKpy1+JQbGB6GuHvF/c4scnzTi7vFEVZzONNA7uAN9zghr2SxGKJ0NJuPPzgkuKReTOTVxu8tWqkxOPmV4D2zEsC1zzbFg63lkxZLVTByc1ZLlQBxzPP+Uj3xblnUICaoXpyJZpvixqc+1baEFMw7sxgmL0Kl6tntRLXKhAi71Rb4yxKd4aVonJnWNnrilhFI9haEhUuUXmLxVai9y7TpQvwxQp13edK5FG/qtBilraLK19vDDPlkCRRb1FB/yjhevk0tfAAvtBVHqSxpNOOYtNM5UxFQUsKzH55Uzh50epPVMvJJIQ8FeNguu4qN26DiE34wL/yHKZvEMZMTjI7T7zU0o9YdP7Su2mzqxoll+nd8BsjqXGc0HB+t59xnYZ+KZnkPheTppW7BnUfgm4hm80fxVJkVJaRUnRMpHeeqMgyOIbWaIpkwLRQrrkm4YYCVwJFDmD6IV3K1BSUxMMXOFOK2E0iiRwL4iOdYt3vJB2l7VyIJIyv5ykKJ44C0/vQMogVgHkMvOmIy7R1cQJXFEHUEsoECoZNsLp15gTZSK/T+WiwIq8OjvYPnR/e7R8evDp4kSnV32N3zBNb8nUZdixVCV+O8fElC8H/c7qH1/Y9rPJ3Bf5/e6dg/7+9t7b/+o+y//r0WEUw91dUHleEfCE0wRa+LNMGK0ZeD6UQlmFB3n63j/Xe2fWao4HbbO4Nt4bus+2dobe3t/f06fmOe/7Ubbr2ln2+83Rvqzl8tnW+29oeuLs73t5gF/567lZzz5pjUjYdR2l0EbvTsRmNTNdMJkCb/4iGZitZlG235tqTPZr+fJ4Z2adp26TlxmKNmwTL2dBwQyVDxq5WAJldWS7EiGLjY+RtjT7d0Gw1/eTDHDIwq6h5xwkfYRiVGUWVGUTJICLmc9N+WvCPLI0EXDSIMla1hpIRMnRzoszaEN4ziQIxqJSlVoAwmMPexw4xfRa38z2cNlpWasl4MziKOMLiu7IzJXbKIzpFAVJvHj+33l8/fXZy9Pp9/D58f22770kRkgVqN3JBRUTX5eKKZPdF8p5VDLOkPCkS2btIlfKlSOYfJGYucErU9C5SCyKoAMsl9HMCuwBj7yK11HqLQ86VOBXzrdy8EtKDJo3KHixKo1YZn18paP/wZZ7aa260DhzQokCw2jHBkycivzUNL9gsgjoGcpLSl2qp6v/18eHL7pGDjMmXrPfP8/8slBuJkVHwwBLAEv6/1Wrm9f87Tfi05v+/UPsPh1/q99BGIFJAyEpZdIAgJQNprs/vDMVDYw+9jdHkUPPq45vlis4lFLlTxcWv617db4LVJ2f08dmsPPg1V8pJTG5zzl17Va2uZCSCaD+7VYh6KlRmHcKdafvM9l71rhUQTCGHPQkdldly6P6zGnAhvWbyizGlFcIoUoCyq+YZWN7IQNSRmxa0ms08RFkd5xtB8Epm6Ohq8nyqveBgRGuHUq3CliuO+Yv76Yvjo7PT40Pn5HD/qPvfsKP+Z+7/mqr9cff/XXtvu7D/22v7zy/J//NB1IOfpv/7fb1CLygcCMjh0BgMhnojVXS/znwPnSKm6E4WuJeepqmbo3pbXXc3xxsUeq7CV3vN/GSfUP705TmELtPoke6qVFe33EtwqznfS/AjVDvlPqIPxx0uVtF9sf6LpU6AxehoWrDbX9Qot8VlUnQNpAW1xClvmVpnFdVOafhWAZrzhCvmyAwotQz8DFtEUjMMoC3kOos0JtPu15BztKt0EWclM+Cgzy3xWYfeqvaNkR/jrb8IX2NxRtjVm4mBhxizqRP4sO7gm71rPjHx+Jr/Ib/Oi+kMZ2GFudExMxvb4rSOUDtDL0hdjNOHbwWfOYDXffjMeg5Seg4WQNUZ10SHZbVAeFUboMJiW1eoCExbDXK1ilAXroZdB31w9Ks1c1XkMtLsMtwZYIlpV4LLmJnK9GJFa0ymHzG5CMPUE+pPzRYmnm8JU8gol2lP8aAtB80cRUXFud1TaRN1d9GsqRky+37I7IXImg6z91D9TssHab6DLqvUffDYc/GI+ihhh5dUp7yL7oHFLsVSphhXbpMVFkzSmZMFJyA6D3/mKdHn2a1+DiV60SZHivTfHbz+DudNiTQvfEgP/tV96fCLIZy8R+la9P/jyf+axdjjyv87drNZlP/X9j9/IPn/484CHtImqFzm59MVrcnLhf7/DJVB/gBCiHF0XXD7004LMNDf7d2nnBm0PsYzNHdyoHzg9aJL0AasbxyK4PkgrqPZyQB7XlAamdPIE4qaqSZLnT3MwDiKUv1YBOEzax4lRhTXsGuIOiVDLk0/AHWPhTHlAY9gFOowLS1CxJKRtfAHAkIz9K+zRVVPYd1YueY4xaOOuTXOty0nHEeX85GLk5ECbjyQWJxJOHneo1oFL8VSfZWmeRtEGGY40HVvD6dhU9xZtygW1u9t2rZQK/Zpi/6z27ctVpjZT5sFM7d5M6zMsC3TipXNOKNEySWW+mcxQivTat3L/mxVZZboI36HQymdW2IltmShFige5dvEm6sond/LsMyOq+hplXmLtZe4inHBi9WdRb2Sl02VKtQ0rd5K5lK5HHmrqTJLo+q9DaWW+BzOEfXmelB8qqx38u7bw4MXztm7o6Pu4fr49ouV/5iLzY3DzoHp0joMZ/8Y/h9be3bB/2Nnr7mW/x5R/rsIotKLvaPkc8t3TEmlhNqkKxFhb8UqNfC/CgbwF7RSXmiNL084UeQyEn7XOBNia5hUVOI+zvdlzMX8iYmNq9K+3iJWDk8m2EHEnSpyUIVFAEjMLlvRSKB3KpfeTSdwJ+dD17xum9fyskKxf/ZkJEhF5csQ0P7dy5feV3OW6aTVzOoZinrMkn0HRqv8VIWhWR+srA9WPuvBygoz9KMOWPgqWPGMRYdeH7N8KccsD3J0sviIhk2de4cv+iMeqQj+j2SzRjL+HDwGMnV7Ozvz7//azd//YNtb6/sfH+X3f77anCXx5rkfbnrhlXnuJmMj8VKz3vVmkYmu/iPXD4jV63wkj2ecvjs6O3jT7Vh/RjTSdN8y6L5ssz41IYXBWCYHCqKLRL4wR/3sPWMC+Ycs0NBgyL9ZBoan6fXMr8x6khWg+AuY/f7fTLTOB1oyvUnHUbgFPMg332yc/LQxn+uVjDGazQ1iL02MaYdY4I2H8mTYgEWvxSNg5TQoEZUhuE9WtjSqs0FUB3JGSWMwnkTDyrRmNqPdZrNqnPwE/B51x1dZQweopCk5z/nbPPXgRtG0027aTJW7UZN6w6plPt8celeb4SwIzNbzv9qyk09+Ovvu+Ohk/+y7ThIPzDAaz6ZZhSbm7MoHwoqh4JX7CBvZfYRtvGarXh9HSWrKWsAHagVWBp5h3tQD78oL8MrsyHyO84hfaQgywQVVyPwrVMYbjCPzz1+Zz5W5weGm/tCiHivMCuWErJY/B8MQD6SfY/eDfY2XADBtpfL1G4ACHpkfxCzSxq7a3aSlJbYbxjfm5y+dDogNbaF/zF+/1oaJnSSGqk1sVVmoJa6AfHuTpN6ke+2nlY1/Hp9+3z11Trv7L39ysJuO351tiG1JS6VNaIPm22eYbM1nTX6Ssdpso3M5oDkVYF9K138VL5f7+eDN/uuus39y4IgM9Nf6yNmKF8ctnKbQjHnTdBABWwEcUBqtMFdVYDFhaSxNW87bFZz7lO5jQzoyN3ifvO2e/qgN7fvw8PjF/qHz7vSwUz4+78PM9arzl+R9uFHa82ut4R9O/5ek0fQzsX/L+L/mbmsrx/8193bX/p9fPv+HwjjGeUR5nB14mApZM9l2/DdzGCGH5g9R7dXJ6MmfMSujfbT5Aa9HnB6H1Pg7yp9tBgKkagkt4KUP+0e9ydIsIL2SLDIcHOCse/pGwPz737CPz7y/gdTKTDQmZn2kIDdQEWYM8W7Wcsr69uz4BMjrD++gs7ovOYndMH6n9Y98w+ehAMvW/7adt/+yt+z1+v9i1r9gR/j6X4ErsQxlPWtEY1Mx+ZIL3BTri59WIoMiuA/B/s1DUrVoAa/3fzdOufnnQ1OBJevf3tvL639aO1vr878vf///9uBI2c8HQTQbjgI3Rhuhw+PXHUUVxFcsikXL1UaK4P65BM8tTfDMNE3XUBFolcZ7DAA3cAaHIPbFHkj+JtQrEnBYXALlXfjpeHbeGEQTpR/ULtnklykD3cSYrOnmMPoQ4imnClQP/HB2XXcnw91tLBkVQSYsPV6aUANJysp7nncucTX9vvnXvyrsUE58VkCrJWwSFNomTQI+kLwKA4mcEBOfeaM5r1evh1HdnaXRbDpEs7o6WvKYpQNgPmeYFsvESuWoqchfOshc/rmSeL+atrnVrHJ2EkpCdvAiBoG53o3MDTEQvbpb/61Zf9b/+n0DBivr3Pc4OBu8RSbwfrAuzLot2MCqwoWGAKVtUIwD5ZvUX5INkV6sPO1JGjg3vVHF8JJtTmbzrv3UbDLe0zSFRoC4UDaSNCAEZcOuh1MJVheF/QT5nGKmP1zk/5XsP1qo7M/R/y17Tf8f0/4jSjTr/yhpwG7gx1HYgM2ARxGvWHllmVVjV7qwu2jq7vkAQC/G/i+XwSSMpr/GSTq7+nB981sdRnR7Z9eqMr36CC1yp34D8w4Cn24AZaWfwZcX9IVBFpRrAnAQe0AyYM5ODWOxjf69Kihs8WlBMPNUxw2HzjQKArqwVZiaMBf60CP7zGrBIP02O362oFTAjn3FGoOnCvh9M51MN0fuJTC87E7aMitFIIE2VIrO6qH1HaXhFcazMSm5gyaxNV6nDvtTMyfuNb/PPfF/8zotxS0+6+kK4CI1MhsKpS1YHH1r0E0+882TZVhNFqWno9pij70giKxVwn9WC/cwN5R43PlbmHm0opgOciuFvJg8x/64FFI67ZMBi9g42CXHO81qzhc+65mi+WyGT+kYPVSSsNwttq+pwaHNMW/fPBNfvWJFf/qc2TDMp1ahr5aVMR++aA3LHB7YlF/ref/wP2X/FzHkH33/bxXlv63t9f0/j7z/l+/5G/k9f6O2cZ8ddQPv/P4Me362SzPRzPETh3sL4H498UPAEojtevHWySL8fezWiBvAhhAQV9nCmkUYRk4x+XaDwW+0zY3ockPjR1j0N5RYfaC7zjnxN+xI7bM1VESCU9vKzqCr+Zahw9fybErE4Q2NZdtob3CO7QPMwYuNu5ICttTuyFgQ6BO8kCdxPOA5bvh1fjQPZuFlCHKyCGx2724az6/lvVbB3ScxXhvljJfSl2PBeG2wxkM1AQ+sVWS6NtrAc23wLoAUwXFtrMBx4f2GD1otYgfnVExeALlqzdb79wPu/59F9l9J/t/Jx/9DB5D1/v+I+386RurshxdlQQCm6NVWujnjlBG78z+i8zduCIkgd/6AktarWRCQC6NKtDGHA/QTMfh0Dc+QCDUUPxg7QuIQdDqIBpfoCypq1ziED1zWA07Vv0IHRrafoqirfFospYdRSKdPHElNya37x2MFdAmLl/F1x7S171r58FLJPtR4ptLAZCCQ7VRXLLSuFio1DXQZlXBPZH4lXGMwuUCSnQ1MZbF6YKskSoDPXHMAU4OpQ9A0mgny4V2VBOWQHEHQYaVmYsQuVYjG2HiQCljaZQIwov3gAlKKXpdZda0qiKqdLvqGbMgDpQlYyCCIEk+GeqaZyBoObCNdHOYNxZy7QFW0Oue6V7hXV5dMKczG2mJXC2P0ceNhl4xH2UDYym5Zlt66y00wtqAbZOuXVPTFmtN0lOHb4vgKPU19AMJDpVra+evd9g+9/4v7Gx9b/m/u7TSL8v/Oev9/jF/Jpj6IwpF/IY3emcfJ25r5svtq/93hmfBBUWmpmDq0s3vXGO2EK85zxJXT7RwqIvJSD6xFaABiImrA5ONy5zEmX2ESOqTyDD3V5ZGkOQDX0mV+zRuugtkadOtLjXA22BUv/IUudOHP+q0tVMMK3WZK/z2rof5eCzlRwSIFbnoWuOmF46bnMtx0pyn9V4LbkvKTBRxOihuvaKqxwvrHaDOOcK14SCKwZP3bW7t5/++trZ11/O8/hP7vIc78PrMCkOYt47QSOsQWc9hB3YMDK8EZp5PPqAvk8W/yZ0JWUYehR8laqBPMhcvqWTzGSp1Fi1KDqRTjS80vWdE0anyejDLVXhRiqqbnEZdJy6hVuXQYvoQfxjYbdqOZTx9Gg4ROQukhl8ijVmEyf8wBsFhQmM7DSenJdGlF21Su3ZDpd2tWtIT+gxwJGw5OogfmAJfR/92tAv+3u6b/j0r/w9lkeoPELZwyGnxycCioLtn4LdcFZdNHZORfQMgmDArJ5ilRzM1+E6DSoTP0uBbZc+KLc5ZLUO2bm5p5fQ0kNpw2/HAIlCqpSE6rWjPpXrwOJM78MN3iR9xuHLMcMMaDy0oFEPzFbO3s1gCdeMKPT0x7z/waPz4xt+wqSwKk7rWfdOp2tQFbF8YM5OifVrN7DAE/9U8DewTKc28q8D/skaevv+VkmK7MAzi9Oyo8GpdsQz4wWSIvHkRqDRAlAOLqwfkQSYqBG55z3YieNgv9X2ce7EJBFCcMCpZf+TiJE5bzAHpSHxq9H0LvQ4UaXzPV8ak0gV+Ff1Vlb9XVED+iGotUEKgISQfjjpVNCEtRSyzrx4X1p7MlZ+hPvBC3pmR5I2iWsPkglVz3aoQs6z6NeKz1z6vvh/7nK2OZ/N/czvH/zT34ren/Y/x6bAL0uck0BV/qmEk8MNzhMJoS4ar/uuaXvtTf9GYaR0gaG2k0CX6X9Q/cXv7+z+bu2v/zkdb/+cwPhvWEHP77hjAuMSmQGMjU0zSKguR5hwVo+jD2vMDqGyzTObACXoiBXBXQBqWRDG4ZRo/Prr5Bfp4Yv3iObFknITLhLiNccsQMTHKEPZ25KvGv+9q9muhuYrK85tCbRHT2dAY7/Y1ff7N/YGqOKiZgYL4q5tGPBy8P9s2zbfO61bBk6+uMGGI5zztbDdu2DJCvPSbj32JMDkx6c3Bm3RkuWYdQh92KNv7v//vf/xtemN+PIxcDcE7QUwCDojbcyd+HbnhxCQmNoXdl3fWhXRhGjkW5QiTAIVhcVfK8A223d0hwtXi4Afy2xQRpa3oDyKALnneYZG4RFy/fpn4QRB+ed2wODmzQYAypDY5x6I9GM+i0BHFuC6DYDRPovQl93+Eyu4VsekCcGOCjj/1scBsRDYsb1NXG9A3cWGgisT3meYfmEDpaXGORrT2MFNbDWdNQ5s8UZhWalDRGwOf3DZhyscemYzyADGuS/aA/4f+HejNhAPrA1z8so/9721uF+J+7W+vzn8eU/3MxPxdahcwPCLq6nvc+2mYxLesrqZ2lvnklLbIawtRx6O5TB++CSKLgCgRcIEYxVDfp2X2jC3tF9+gFXkZRGmEO2lyXVWUK3joqeBMWaU7m5zgb5MVY4QVw9w2Ku+tElx12QbC4vSLrrF6xh/r3uqbix/3Dg5e5aPmi1iQzK54a84PZMxEbrR0c2Csc1toKV4wr41zjYbjbZpLG2LwpbBcee62a9ecm3nDRXuGuxJZyU2Ih+ZuyQOqlSvlR5qlxy2p2Z827NRHDKc3X17P7GXStum61Ihtbcl/GHEMgbvDBYjDtk64GBpKFgR9haHNT1Noc+kM64yO7KfNWFHZnCf0HMB2o6Nf6eLlpyznaHkGKw21cHHHRQFvtFyw4s3cRZjs71aW3M+prmdUHsw9NALGKdjMWi/ymu0xpOITvVOZu1bTMJ+bu9nx/Km6Mo4wpvyv8nk5Wub5ayXxn1TOlfAV71sRPEixMXky+mmcWmd/SotfMqRcU5IekmyovqOBsNLfkAqQ0+55Lpkg3Z90Vs2aN0NNWbdJ54IaX3Cz8E5qkEQph2fzkCauYqRBURHL3UTWdhclsiruUN2QH+Z+ztoKwZ2YDH1tpZmQPtEmtr+4Z508uHD6H8yRXK0Yrh+I639sLUWkwm/llpoeEepF/YRbImu4qYfCC0GvegrSwS/fCmsTAoseG2PrqosVHPAuLCC2tYfOdWpgD6rYmCryTd3eo3bJgfPVaEcH8lJ5XplqJGaJm4oNFLRsJvZsYlR3NguDzLRBp7LhkTSyygqRvs9C9cv3APQ+8jzM8UBCUbhZ63/B4mUqmxX10//VkLJhAghExVQMDffNqo6eQdtuxsuFg4paSqNIWTNtWM5YuFoR6pkBpxB/dj1sa+hy9zQMoMw2TVMQlHd02d0Tt74oXDQleI+MNa1qKkV0voHdfEs1iZpoxmM4cLmBl4jpj2xXwQcTsMpRgag6qXbIsOMWAt3IuAJ9kkzQcSr0wBrbyqkAV429nTZfyzqdfTirDYSxEoMJZKB+9OD46O91/cVYSVJvCoDgOKuwch8wPHQc5ZsfhDtWMfTb+a/Q/MJJ4/YuQaB/UAmSx/sdutex8/Mcte2d9/+fvpf+JvRXvfFmmMCnTlBhvv9sHQY1uG8GAOlP0yoyt/+k168/c+qh/u7t992dYoCeH+y+63x0fvuyeoup84dVfXOKlK19wRTP1AlvH/ECzC5XAJJUoY1LDTxzmGAqsmtimTFFbE5PaJun17yxDkUyVG+IITXZ7aMlFM6x6rFQHBNUKeexwnQhsHF5QphDhtWT91cAdiAwKWGaqLt84EaSOfYqHHreE7w4qTXBfxXfWAoWAFw5nMep5oYuo+yz5pb5D/oCKOSFdb5dkkPhaJ/tuN75RAS+iALpPArJXunsBeartlgordU8CeokajeWi2xxklnl3O7AJQE7hEnbJ9Wr8mkNxeQnLw+7G4gaWdZ4qMmQ8oMMPhio42Sz65P/m1ZkKo5Fep9ots+W7HGAGngJn/Pvw1cHR/qHzwztguV4dvNg/Ozg+4vGfrMLdLYSzAYPHLEIZGpgkZdUwxTRHZLxES2kGNky/BDErotc263i5HS8AL2XsV4u1YSg6ytUZZfXAmypNPIXCa2qJ5CzoaUMthU2pnuQ/mJsY7e1lYNmVSZqjAenwBAy/kIkcizWDfQHAjbFqMocwvsq+kPW++kE34bf6KzgIiKxD5VbY8xGMQ2rvqo0jQiILFlcRmny1Uft0kzLWmT0VoK/bAWdKtAJelVsvKtUE+Nw7FNkFibBMA5KP1Tzq134pa1kYbMFlqs2TdAtr4ae+y3B6XBnbk9J3TcYBWvykeYTglFdKQMrIB3PHEK6OyIdek8IbHS+8cDah89KSfPnoCjSUzHtTjuMoo8S9W0J9158zoExEkeDsMstiFyzooVXRZtcxsU6lOs+7k0l2RK75K5dWtrjZHgxSGzrZItnvsUt5+tm24GTkm0OIq3wKQ4pZaRS31JQefC5cszYjL1fMsPota6qjqlqzXnYbFkO2pMAFiPIjQpjm3pK1Gk41khNDqN+elceiTH6elk1+6uHS+a4hZswXDlZurucuqANswFnW2ENrUSPm3Oc1/w6zeVd6LS7DvncZ9r3LKN7+VXKJ3KILwJY14Z7o7XuhL9wVtkrlV+/++yG3V0a+hLOQcCybxmBomBjFCvwrj2uG2WHTpXfTNolD7zXZsoIvNfaFVhDx2j1rMPYGl7B+G37qTZJKVdMq5bA+kMbto/VlWt1EA1jMIjkG+fvby7qUsjpCsCgyKUK7Rc1nFmQODymMIRsTL12gyNL6TAFT69mmuq2g5tI0no4iyqxVn1kIKN4rPaHF5CEd5ioy52SmURfhIO6n1pyDUZ0Y2om9xFeecVXSoDaNAUu8mhqCJFIVLV0szyaaab39/uDkpPuSH835oxJ4WY1C2C4Bq9yjTuCKdqVsbxbZFLGCY2Bys1AI4I1R5UWywz9hF4iX6ioLfg40LMFMBs93pl4+NYJ9MrgaIZ7ilfSkQm78EvnhUr2MqV423EUCG6NBTpX6WFMN6U7WWlBoVGFJZ2tWCRWYWWg0Es+NB+NKrJ98t98nT9jJ9/vk695+/WcWeLrh1Pu3rWYNLVMYxvIeYTvfXCa5oC+/tchFn99O35uy9pc1Pmt5H1UIeep3N0/nLjT+zo/d04NXP6117evf+rf+rX/r3/q3/q1/69/6t/6tf+vf+rf+rX/r3/r3O//+PxfeVPsAQAEA"

payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest() == PAYLOAD_SHA256
manifest_bytes = json.dumps(MANIFEST, sort_keys=True, separators=(',', ':')).encode()
assert hashlib.sha256(manifest_bytes).hexdigest() == MANIFEST_SHA256

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True)
with tarfile.open(fileobj=io.BytesIO(payload), mode='r:gz') as tf:
    tf.extractall(WORKSPACE, filter='data')

for item in MANIFEST:
    path = WORKSPACE / item['path']
    data = path.read_bytes()
    assert len(data) == item['bytes']
    assert hashlib.sha256(data).hexdigest() == item['sha256']

for name in ['outputs','metadata','evidence','logs','.runtime']:
    (WORKSPACE/name).mkdir(parents=True, exist_ok=True)
assert not any((WORKSPACE/'evidence').iterdir())
assert not any((WORKSPACE/'outputs').iterdir())
assert not any((WORKSPACE/'metadata').iterdir())
print(f'BOOTSTRAP_FILES={len(MANIFEST)}')
print('BOOTSTRAP_INTEGRITY=PASS')


## Step 3 — Static qualification before GPU load / Kiểm tra tĩnh trước khi nạp GPU

**English**

Run unit/contract tests, Python compilation, shell syntax validation, and notebook/source checks before loading the resident model.

**Tiếng Việt**

Chạy unit/contract test, compile Python, kiểm tra cú pháp shell và các static check trước khi nạp resident model.

### Expected evidence / Kết quả cần thấy

**English**

All test commands succeed and the cell ends with `STATIC_QUALIFICATION=PASS`.

**Tiếng Việt**

Tất cả lệnh test/validation đều thành công và cell kết thúc bằng `STATIC_QUALIFICATION=PASS`.


In [ ]:
import os, subprocess, sys
os.chdir(WORKSPACE)
env = dict(os.environ, PYTHONPATH=str(WORKSPACE/'src'))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True, env=env)
subprocess.run([sys.executable, 'scripts/api_contract_acceptance.py'], check=True, env=env)
subprocess.run([sys.executable, '-m', 'compileall', '-q', 'src', 'scripts', 'tests'], check=True, env=env)
for script in (WORKSPACE/'scripts').glob('*.sh'):
    subprocess.run(['bash','-n',str(script)], check=True)
print('STATIC_QUALIFICATION=PASS')


## Step 4 — Start the resident BF16 T4×2 worker / Khởi động resident BF16 worker trên T4×2

**English**

Start the internal loopback worker and wait until one BF16 pipeline is resident with the qualified component placement across both T4 GPUs.

**Tiếng Việt**

Khởi động internal loopback worker và chờ một BF16 pipeline resident với component placement đã qualification trên cả hai T4.

### Expected evidence / Kết quả cần thấy

**English**

Readiness JSON shows transformer on GPU0, text encoder + VAE on GPU1, dtype `bfloat16`, then `WORKER_READY=PASS`.

**Tiếng Việt**

Readiness JSON cho thấy transformer trên GPU0, text encoder + VAE trên GPU1, dtype `bfloat16`, sau đó là `WORKER_READY=PASS`.


In [ ]:
import os, secrets, subprocess, sys, time, urllib.request, json
RUNTIME = WORKSPACE/'.runtime'
TOKEN_FILE = RUNTIME/'api_token'
TOKEN_FILE.write_text(secrets.token_urlsafe(32))
os.chmod(TOKEN_FILE, 0o600)
os.environ['ZIMAGE_API_TOKEN'] = TOKEN_FILE.read_text().strip()
env = dict(os.environ, PYTHONPATH=str(WORKSPACE/'src'))
worker_log = open(WORKSPACE/'logs/worker.log','w')
worker = subprocess.Popen([sys.executable,'-m','uvicorn','zimage_server.worker_api:app','--host','127.0.0.1','--port','8101','--log-level','info'], cwd=WORKSPACE, env=env, stdout=worker_log, stderr=subprocess.STDOUT)
(RUNTIME/'worker.pid').write_text(str(worker.pid))

deadline=time.monotonic()+420
while True:
    if worker.poll() is not None:
        raise RuntimeError((WORKSPACE/'logs/worker.log').read_text()[-8000:])
    try:
        with urllib.request.urlopen('http://127.0.0.1:8101/ready', timeout=5) as r:
            ready=json.load(r); break
    except Exception:
        if time.monotonic()>deadline: raise TimeoutError('worker readiness')
        time.sleep(2)
expected={'transformer':0,'text_encoder':1,'vae':1}
assert ready['load']['device_map'] == expected, ready
assert ready['load']['dtype'] == 'bfloat16'
(WORKSPACE/'evidence/runtime-ready.json').write_text(json.dumps(ready, indent=2))
print(json.dumps(ready, indent=2))
print('WORKER_READY=PASS')


## Step 5 — Start the authenticated REST coordinator / Khởi động REST coordinator có xác thực

**English**

Start the CPU-side FastAPI coordinator on port 8090, validate the public health/root routes and authenticated readiness/info routes, and confirm only qualified profiles are exposed.

**Tiếng Việt**

Khởi động FastAPI coordinator phía CPU trên cổng 8090, kiểm tra health/root public cùng readiness/info có xác thực, và xác nhận chỉ expose các profile đã qualification.

### Expected evidence / Kết quả cần thấy

**English**

The runtime info JSON is printed and the cell ends with `COORDINATOR_READY=PASS`.

**Tiếng Việt**

Runtime info JSON được in ra và cell kết thúc bằng `COORDINATOR_READY=PASS`.


In [ ]:
coord_log = open(WORKSPACE/'logs/coordinator.log','w')
coord = subprocess.Popen([sys.executable,'-m','uvicorn','zimage_server.app:app','--host','127.0.0.1','--port','8090','--log-level','info'], cwd=WORKSPACE, env=env, stdout=coord_log, stderr=subprocess.STDOUT)
(RUNTIME/'coordinator.pid').write_text(str(coord.pid))

def get(path, auth=False):
    headers={'Authorization':'Bearer '+os.environ['ZIMAGE_API_TOKEN']} if auth else {}
    req=urllib.request.Request('http://127.0.0.1:8090'+path, headers=headers)
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.status, json.load(r)

deadline=time.monotonic()+30
while True:
    try:
        status, health=get('/health'); break
    except Exception:
        if coord.poll() is not None: raise RuntimeError((WORKSPACE/'logs/coordinator.log').read_text()[-8000:])
        if time.monotonic()>deadline: raise TimeoutError('coordinator readiness')
        time.sleep(1)
assert status==200 and health['status']=='ok'
status, root=get('/')
assert root['type']=='rest-api'
status, ready=get('/ready', auth=True)
assert ready['ready'] is True
status, info=get('/v1/info', auth=True)
assert set(info['profiles']) == {'safe_512','high_768'}
assert 'high_1024' not in info['profiles']
print(json.dumps(info, indent=2))
print('COORDINATOR_READY=PASS')


## Step 6 — Run deterministic 512×512 acceptance / Chạy acceptance 512×512 xác định

**English**

Submit the golden `safe_512` request through the REST API and verify the output checksum against the accepted deterministic image.

**Tiếng Việt**

Gửi golden request `safe_512` qua REST API và đối chiếu checksum output với ảnh deterministic đã được chấp nhận.

### Expected evidence / Kết quả cần thấy

**English**

`GOLDEN_512_DETERMINISM=PASS`; expected PNG SHA-256 is `56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307`.

**Tiếng Việt**

`GOLDEN_512_DETERMINISM=PASS`; SHA-256 PNG mong đợi là `56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307`.


In [ ]:
subprocess.run([sys.executable,'scripts/golden_rest_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
golden=json.loads((WORKSPACE/'metadata/golden-512-seed42.json').read_text())
assert golden['sha256']=='56e0fca007d3da945de77788b5ab8a0a131b58730d93b624ca65e7c6ca6ea307'
print('GOLDEN_512_DETERMINISM=PASS')


## Step 7 — Run five-job queue endurance / Chạy endurance queue 5 job

**English**

Exercise five sequential generation jobs through the bounded single-flight queue and record timing/queue evidence.

**Tiếng Việt**

Chạy năm generation job tuần tự qua bounded single-flight queue và ghi lại timing/queue evidence.

### Expected evidence / Kết quả cần thấy

**English**

The endurance record is printed and the cell ends with `QUEUE_ENDURANCE=PASS`.

**Tiếng Việt**

Endurance record được in ra và cell kết thúc bằng `QUEUE_ENDURANCE=PASS`.


In [ ]:
subprocess.run([sys.executable,'scripts/endurance_queue_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
endurance=json.loads((WORKSPACE/'evidence/endurance-5jobs.json').read_text())
print(json.dumps(endurance, indent=2))
print('QUEUE_ENDURANCE=PASS')


## Step 8 — Verify authentication and error boundaries / Xác minh authentication và error boundary

**English**

Exercise rejected requests for missing/invalid auth, invalid inputs, unsupported profiles/resolutions, queue pressure, worker unavailability, and premature image retrieval.

**Tiếng Việt**

Kiểm tra các request bị từ chối khi thiếu/sai auth, input không hợp lệ, profile/resolution không hỗ trợ, queue pressure, worker unavailable và lấy ảnh quá sớm.

### Expected evidence / Kết quả cần thấy

**English**

The boundary matrix is printed, 1024×1024 is rejected with HTTP 422, and the cell ends with `ERROR_BOUNDARIES=PASS`.

**Tiếng Việt**

Boundary matrix được in ra, 1024×1024 bị từ chối bằng HTTP 422 và cell kết thúc bằng `ERROR_BOUNDARIES=PASS`.


In [ ]:
subprocess.run([sys.executable,'scripts/error_boundary_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
errors=json.loads((WORKSPACE/'evidence/error-boundary.json').read_text())
assert errors['checks']['unsupported_1024'][0] == 422
print(json.dumps(errors, indent=2))
print('ERROR_BOUNDARIES=PASS')


## Step 9 — Run mixed 512 → 768 → 512 qualification / Chạy qualification hỗn hợp 512 → 768 → 512

**English**

Prove that one resident pipeline can serve the qualified low/high/low profile sequence without replacing the model or creating concurrent GPU replicas.

**Tiếng Việt**

Chứng minh một resident pipeline có thể phục vụ chuỗi profile thấp/cao/thấp đã qualification mà không thay model hoặc tạo GPU replica chạy đồng thời.

### Expected evidence / Kết quả cần thấy

**English**

The mixed-profile evidence is printed and the cell ends with `MIXED_PROFILE_QUALIFICATION=PASS`.

**Tiếng Việt**

Mixed-profile evidence được in ra và cell kết thúc bằng `MIXED_PROFILE_QUALIFICATION=PASS`.


In [ ]:
subprocess.run([sys.executable,'scripts/mixed_profile_acceptance.py'], cwd=WORKSPACE, env=env, check=False)
# The first 512 request can legitimately warm allocator state; qualify from captured results.
subprocess.run([sys.executable,'scripts/qualify_mixed_existing.py'], cwd=WORKSPACE, env=env, check=True)
mixed=json.loads((WORKSPACE/'evidence/mixed-512-768-512.json').read_text())
print(json.dumps(mixed, indent=2))
print('MIXED_PROFILE_QUALIFICATION=PASS')


## Step 10 — Optional public HTTPS transport check / Kiểm tra public HTTPS transport tùy chọn

**English**

Optionally expose the coordinator through a temporary Cloudflare Quick Tunnel and run an external REST acceptance check. This is transport evidence only, not runtime authority.

**Tiếng Việt**

Tùy chọn expose coordinator qua Cloudflare Quick Tunnel tạm thời và chạy external REST acceptance. Đây chỉ là transport evidence, không phải runtime authority.

### Expected evidence / Kết quả cần thấy

**English**

Either `PUBLIC_HTTPS_REST_ACCEPTANCE=PASS` or the explicit `PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED` marker.

**Tiếng Việt**

Nhận một trong hai marker rõ ràng: `PUBLIC_HTTPS_REST_ACCEPTANCE=PASS` hoặc `PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED`.


In [ ]:
RUN_PUBLIC_TUNNEL = False
public_evidence = WORKSPACE / 'evidence/public-tunnel-acceptance.json'

if not RUN_PUBLIC_TUNNEL:
    skipped = {
        'verdict': 'SKIPPED',
        'optional': True,
        'reason': 'Set RUN_PUBLIC_TUNNEL=True to run the external Quick Tunnel gate.'
    }
    public_evidence.write_text(json.dumps(skipped, indent=2))
    print(json.dumps(skipped, indent=2))
    print('PUBLIC_HTTPS_REST_ACCEPTANCE=SKIPPED')
else:
    CF_VERSION = '2026.10.0'
    CF = RUNTIME / 'cloudflared'
    if not CF.exists():
        url = f'https://github.com/cloudflare/cloudflared/releases/download/{CF_VERSION}/cloudflared-linux-amd64'
        urllib.request.urlretrieve(url, CF)
        os.chmod(CF, 0o755)
    tunnel_log = open(WORKSPACE / 'logs/cloudflared.log', 'w')
    tunnel = subprocess.Popen([
        str(CF), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8090'
    ], stdout=tunnel_log, stderr=subprocess.STDOUT)
    (RUNTIME / 'tunnel.pid').write_text(str(tunnel.pid))
    import re
    deadline = time.monotonic() + 30
    public_url = None
    while time.monotonic() < deadline:
        time.sleep(1)
        text = (WORKSPACE / 'logs/cloudflared.log').read_text(errors='replace')
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if m:
            public_url = m.group(0)
            break
    if public_url is None:
        raise RuntimeError('Quick Tunnel URL not established')
    (RUNTIME / 'tunnel_url').write_text(public_url)
    subprocess.run([sys.executable, 'scripts/public_tunnel_acceptance.py'], cwd=WORKSPACE, env=env, check=True)
    print('PUBLIC_HTTPS_REST_ACCEPTANCE=PASS')


## Step 11 — Finalize the qualification record / Hoàn tất qualification record

**English**

Aggregate the current-run evidence into the machine-readable summary and require a final PASS verdict. The local REST API remains available until you explicitly stop it.

**Tiếng Việt**

Tổng hợp evidence của run hiện tại thành summary machine-readable và bắt buộc final verdict là PASS. Local REST API vẫn hoạt động cho đến khi bạn chủ động dừng.

### Expected evidence / Kết quả cần thấy

**English**

`FINAL_QUALIFICATION=PASS`. To stop the services after review, run `bash scripts/stop.sh` from the materialized workspace.

**Tiếng Việt**

`FINAL_QUALIFICATION=PASS`. Sau khi review xong, có thể dừng service bằng `bash scripts/stop.sh` từ materialized workspace.


In [ ]:
subprocess.run([sys.executable,'scripts/finalize_evidence.py'], cwd=WORKSPACE, env=env, check=True)
subprocess.run([sys.executable,'scripts/verify_evidence.py'], cwd=WORKSPACE, env=env, check=True)
summary=json.loads((WORKSPACE/'evidence/qualification-summary.json').read_text())
assert summary['final_verdict']=='PASS', summary
notebook_identity={
    'api_only': True,
    'payload_sha256': PAYLOAD_SHA256,
    'payload_manifest_sha256': MANIFEST_SHA256,
    'model_variation':'dangkhoa2016/tongyi-mai-z-image-turbo/PyTorch/default/1',
    'upstream_revision':'f332072aa78be7aecdf3ee76d5c247082da564a6',
}
print(json.dumps({'qualification':summary,'notebook':notebook_identity}, indent=2))
print('FINAL_QUALIFICATION=PASS')
print('REST API remains on http://127.0.0.1:8090')
print('Bearer token remains private in .runtime/api_token; do not publish it.')
print('Stop services with: bash scripts/stop.sh')
